# Part 2 — Binance → Kafka producer (Spark)

This notebook runs a **Spark job** that polls the public Binance REST API every `POLL_INTERVAL_SECONDS`, normalises each tick into `{symbol, price, volume, timestamp}`, and publishes one JSON message per symbol to the `financial_data` Kafka topic.

We run Spark in **local mode** (`local[*]`) inside this Jupyter container because the Jupyter image uses Python 3.11 while the standalone Spark workers (`apache/spark:3.5.0`) use Python 3.8 — PySpark refuses to run if driver and executors have different Python minor versions. For a lightweight producer that only builds tiny DataFrames (5 rows per iteration) this is the right choice anyway: the remote cluster is reserved for heavier jobs submitted via `spark-submit` from the separate `lab3-spark-client` container (which has matching Python 3.8).

Kafka is reached via the in-network listener `kafka:9092`. The Spark-Kafka connector JAR is pulled from Maven Central via `spark.jars.packages` the first time the Spark session is created (~30 s, then cached in `~/.ivy2`).

**Stop the producer** by interrupting the kernel (the ■ square button in the toolbar).

## 1. Configuration

In [ ]:
SYMBOLS = ["BTCUSDT", "ETHUSDT", "SOLUSDT", "BNBUSDT", "ADAUSDT"]
KAFKA_BROKERS = "kafka:9092"
TOPIC = "financial_data"
POLL_INTERVAL_SECONDS = 5
BINANCE_URL = "https://api.binance.com/api/v3/ticker/24hr"

## 2. Build a Spark session (local mode)

`spark.jars.packages` pulls the Spark-Kafka connector from Maven Central the first time this cell runs (~30 s). Subsequent runs are instant thanks to the Ivy cache.

In [ ]:
import findspark
findspark.init()

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("BinanceToKafkaProducer")
    .master("local[*]")
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark", spark.version, "ready")

## 3. Schema + helper for one REST call

In [ ]:
import requests
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType

SCHEMA = StructType([
    StructField("symbol", StringType(), False),
    StructField("price", DoubleType(), False),
    StructField("volume", DoubleType(), False),
    StructField("timestamp", LongType(), False),
])

def fetch_ticker(symbol: str) -> dict:
    r = requests.get(BINANCE_URL, params={"symbol": symbol}, timeout=5)
    r.raise_for_status()
    d = r.json()
    return {
        "symbol": d["symbol"],
        "price": float(d["lastPrice"]),
        "volume": float(d["volume"]),
        "timestamp": int(d["closeTime"]),
    }

fetch_ticker("BTCUSDT")

## 4. The polling loop
Each iteration:
1. Fetches one REST response per symbol.
2. Builds a Spark DataFrame.
3. Rewrites it to a `(key, value)` shape (the two columns Kafka expects), where `key = symbol` (same symbol always lands on the same partition, preserving per-symbol order).
4. Writes the batch to the `financial_data` topic.

Interrupt the kernel to stop.

In [ ]:
import time
from pyspark.sql.functions import col, struct, to_json

iteration = 0
print(f"Producer started -> topic={TOPIC}, symbols={SYMBOLS}, interval={POLL_INTERVAL_SECONDS}s")
while True:
    iteration += 1
    records = []
    for sym in SYMBOLS:
        try:
            records.append(fetch_ticker(sym))
        except Exception as exc:
            print(f"[iter {iteration}] fetch failed for {sym}: {exc}")

    if not records:
        time.sleep(POLL_INTERVAL_SECONDS)
        continue

    df = spark.createDataFrame(records, schema=SCHEMA)
    out = df.select(
        col("symbol").cast("string").alias("key"),
        to_json(struct("symbol", "price", "volume", "timestamp")).alias("value"),
    )
    (
        out.write
        .format("kafka")
        .option("kafka.bootstrap.servers", KAFKA_BROKERS)
        .option("topic", TOPIC)
        .save()
    )
    sample = records[0]
    print(f"[iter {iteration}] published {len(records)} ticks "
          f"(sample: {sample['symbol']}={sample['price']} vol={sample['volume']})")
    time.sleep(POLL_INTERVAL_SECONDS)

## 5. Clean up
After interrupting the loop above, run this cell to release the Spark session.

In [ ]:
spark.stop()